In [37]:
from dotenv import load_dotenv
load_dotenv()

True

In [38]:
from google import genai
gemini_client = genai.Client() # picks up the API key from the env variable GEMINI_API_KEY

In [39]:
def llm(prompt):
    response = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text

In [40]:
question = 'I have just discovered the course, can I join it now?'
answer = llm(question)
print(answer)

Welcome! To give you an exact answer, I need a little more context. **Which course or platform are you referring to?**

However, here is how it generally works depending on the type of course:

1. **Self-Paced Online Courses (e.g., Udemy, Coursera, edX, YouTube tutorials):** 
   * **Yes, absolutely!** You can enroll and start learning immediately at any time.
2. **Live / Cohort-Based Courses (with set start dates and live sessions):** 
   * **It depends.** If the registration deadline hasn't passed, you can join right away. If it started recently, you might still be able to join and catch up on recorded sessions. 
3. **University or School Classes:** 
   * You will need to check the institution's **add/drop deadline** or contact the instructor directly to request permission for late entry.

**What to do next:**
* If you are looking at a specific website, look for an **"Enroll," "Register,"** or **"Start Course"** button.
* If you reply here with the **name of the course or the platform

In [41]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [42]:
prompt = f"""
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [43]:
answer = llm(prompt)
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.


In [44]:

def rag(question):
    """
    Retrieval-Augmented Generation
    1. Retrieve relevant search results
    2. Build a prompt with the retrieved documents and the question
    3. Generate an answer using the LLM
    """
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [45]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [46]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 471},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 253},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 118}]

In [47]:
documents = [] # list of questions from all courses
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url) # fetch json file of all questions in the course, from its url
    course_response.raise_for_status() # raise an error if the request was unsuccessful
    course_data = course_response.json() # parse the json response into a Python object (list of questions)

    documents.extend(course_data)

len(documents)

1380

In [48]:
from minsearch import Index

index = Index(
    text_fields = ["question", "section", "answer"], # fields for searching
    keyword_fields = ["course"] # existing fields in the dataset for filtering
)

index.fit(documents)

In [49]:
def search(question, course='llm-zoomcamp'):
    """
    Search for relevant documents based on the question and the specified course.
    """
    boost_dict = {"question": 2.0, "section": 0.5} # importance of fields for scoring
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict = boost_dict,
        filter_dict = filter_dict,
        num_results = 5
    )

In [50]:
search_results = search(question)

In [51]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [52]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [53]:
def build_context(search_results):
    """ 
    Build a context dictionary from the search results to be used in the prompt for the LLM.
    """
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [54]:
context = build_context(search_results)
print(context)

USER_PROMPT_TEMPLATE.format(question=question, context=context) # setting the question and context in the prompt template for the LLM

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

To get the certificate, you need to finish a capstone project and complete the
required peer reviews. Homework is not required. You can work through the
material and prepare your project in self-paced

'\nQuestion:\nI have just discovered the course, can I join it now?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Certificate: Can I follow the course in a self-paced mode and get a certificate?\nA: No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Ho

In [55]:
def build_prompt(question, search_results):
    """
    Build a prompt for the LLM using the question and the search results.
    """
    context = build_context(search_results)
    
    # setting the question and context in the prompt template for the LLM
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip() # remove \n at the beginning and end of the prompt

In [56]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I have just discovered the course, can I join it now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

To get the certificate, you need to finish a capstone project and complete the
required peer reviews. Homework is not required